# exp-006: 5세트 fusion 가중치 × 5관점 라벨 — 25셀 NDCG@10 매트릭스

- **목적:** 단일 fusion 가중치 → 관점별 5세트 분기(A·B·C·D·E)의 효과 정량 검증.
  - 핵심 가설 H1: 관점 i 가중치 × 관점 i 라벨 = **대각선 1위** (관점 i 가중치가 그 관점에서 최선)
  - 핵심 가설 H2: D 관점에서 industry=0.55 격상이 단일(0.10) 대비 NDCG@5/10 회복
  - 핵심 가설 H3: 평균 NDCG@10에서 5세트 분기 > 단일 가중치
- **차별성 축:** ③ 양방향(시나리오 비대칭) + ④ GT 부재(관점 분리 라벨링)
- **데이터:** `raw/data/gemini_profile_outputs/`
  - `benchmark_labeled_100_{A,B,C,D,E}.csv` — 관점별 100쌍 라벨 (judge_relevance 0~4)
  - `weighted_results.csv` — 6요소 fusion components (role_semantic, hard_skill, competency, achievement, industry, quality_adjustment)
- **출력:** `raw/experiments/exp-006-25cell-matrix/` (NDCG@10/5 매트릭스 CSV)
- **관련 위키:** 관점별-fusion-가중치-설계 §4, 100쌍-벤치마크-결과분석, dual-encoder-진화-실험계획 exp-006
- **작성일/실행일:** 2026-05-18
- **풀데이터 필요?** ❌ 불필요 — 기존 1k 샘플 weighted_results.csv + 500쌍 라벨만 사용

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json

DATA = Path('raw/data/gemini_profile_outputs')
OUT_DIR = Path('raw/experiments/exp-006-25cell-matrix')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('DATA:', DATA)
print('OUT:', OUT_DIR)

DATA: raw/data/gemini_profile_outputs
OUT: raw/experiments/exp-006-25cell-matrix


In [2]:
# 5관점 라벨 로드
labels = {}
for p in 'ABCDE':
    f = DATA / f'benchmark_labeled_100_{p}.csv'
    df = pd.read_csv(f, encoding='utf-8-sig')
    labels[p] = df
    n_users = df['userId'].nunique()
    n_rows = len(df)
    rel_dist = df['judge_relevance'].value_counts().sort_index().to_dict()
    print(f'관점 {p}: {n_rows}쌍 / {n_users}명 / 라벨분포 {rel_dist}')

관점 A: 100쌍 / 10명 / 라벨분포 {0: 18, 1: 1, 2: 4, 3: 27, 4: 50}
관점 B: 100쌍 / 10명 / 라벨분포 {0: 16, 1: 10, 2: 21, 3: 53}
관점 C: 100쌍 / 10명 / 라벨분포 {0: 16, 1: 7, 2: 2, 3: 20, 4: 55}
관점 D: 100쌍 / 10명 / 라벨분포 {0: 15, 1: 21, 4: 64}
관점 E: 100쌍 / 10명 / 라벨분포 {0: 8, 1: 8, 2: 26, 3: 25, 4: 33}


In [3]:
# fusion 6요소 점수 로드 (weighted_results.csv)
wr = pd.read_csv(DATA / 'weighted_results.csv', encoding='utf-8-sig')
# BOM 제거
wr.columns = [c.lstrip('\ufeff') for c in wr.columns]
fusion_cols = ['role_semantic', 'hard_skill', 'competency', 'achievement', 'industry', 'quality_adjustment']
print(f'weighted_results: {len(wr):,} rows, models={wr["model"].unique()}')
print(f'fusion 컬럼 존재 확인: {[c in wr.columns for c in fusion_cols]}')
wr[['userId', 'job_id'] + fusion_cols].head(3)

weighted_results: 123,573 rows, models=<StringArray>
['profile_weighted_score']
Length: 1, dtype: str
fusion 컬럼 존재 확인: [True, True, True, True, True, True]


,userId,job_id,role_semantic,hard_skill,competency,achievement,industry,quality_adjustment
0,P0001,217234,0.880039,0.333333,0.181818,0.666667,0.0,1.0
1,P0001,258074,0.871483,0.333333,0.100000,0.666667,0.0,1.0
2,P0001,227795,0.860019,0.333333,0.090909,0.666667,0.0,1.0


In [4]:
# 5관점 라벨 vs fusion components 매칭률 확인
for p in 'ABCDE':
    L = labels[p].copy()
    L.columns = [c.lstrip('\ufeff') for c in L.columns]
    # job_id 타입 일치
    L['job_id'] = L['job_id'].astype(str)
    wr['job_id'] = wr['job_id'].astype(str)
    M = L.merge(
        wr[['userId', 'job_id'] + fusion_cols],
        on=['userId', 'job_id'], how='left', suffixes=('_label', '')
    )
    coverage = M['role_semantic'].notna().sum()
    print(f'관점 {p}: {coverage}/{len(L)} 쌍 fusion 6요소 매칭')
    if coverage < len(L):
        missing = M[M['role_semantic'].isna()][['userId', 'job_id']].head(3)
        print(f'  미매칭 샘플:\n{missing}')

관점 A: 34/100 쌍 fusion 6요소 매칭
  미매칭 샘플:
  userId job_id
1  P0493  87281
3  P0493  97157
4  P0493  56482


관점 B: 33/100 쌍 fusion 6요소 매칭
  미매칭 샘플:
  userId  job_id
0  P0369   56482
1  P0369  146480
2  P0369  244657
관점 C: 27/100 쌍 fusion 6요소 매칭
  미매칭 샘플:
  userId  job_id
0  P1001  163698
1  P1001  290183
2  P1001  161864
관점 D: 24/100 쌍 fusion 6요소 매칭
  미매칭 샘플:
  userId  job_id
0  P0679  226654
1  P0679   37446
2  P0679   24451
관점 E: 32/100 쌍 fusion 6요소 매칭
  미매칭 샘플:
  userId  job_id
0  P0522  303209
1  P0522  256520
2  P0522  153326


In [5]:
# 5세트 fusion 가중치 정의 (관점별-fusion-가중치-설계 §4 v0)
WEIGHTS = {
    'A': {'role_semantic': 0.50, 'hard_skill': 0.15, 'competency': 0.10, 'achievement': 0.05, 'industry': 0.10, 'quality_adjustment': 0.10},
    'B': {'role_semantic': 0.20, 'hard_skill': 0.15, 'competency': 0.10, 'achievement': 0.35, 'industry': 0.10, 'quality_adjustment': 0.10},
    'C': {'role_semantic': 0.15, 'hard_skill': 0.45, 'competency': 0.15, 'achievement': 0.05, 'industry': 0.10, 'quality_adjustment': 0.10},
    'D': {'role_semantic': 0.15, 'hard_skill': 0.10, 'competency': 0.05, 'achievement': 0.05, 'industry': 0.55, 'quality_adjustment': 0.10},
    'E': {'role_semantic': 0.20, 'hard_skill': 0.20, 'competency': 0.15, 'achievement': 0.15, 'industry': 0.20, 'quality_adjustment': 0.10},
    'SINGLE': {'role_semantic': 0.35, 'hard_skill': 0.20, 'competency': 0.15, 'achievement': 0.10, 'industry': 0.10, 'quality_adjustment': 0.10},
}
# 합 검증
for k, v in WEIGHTS.items():
    s = sum(v.values())
    assert abs(s - 1.0) < 1e-9, f'{k} 합={s}'
    print(f'{k}: 합={s:.2f}, 시그니처 요소={max(v, key=v.get)}={v[max(v, key=v.get)]:.2f}')

A: 합=1.00, 시그니처 요소=role_semantic=0.50
B: 합=1.00, 시그니처 요소=achievement=0.35
C: 합=1.00, 시그니처 요소=hard_skill=0.45
D: 합=1.00, 시그니처 요소=industry=0.55
E: 합=1.00, 시그니처 요소=role_semantic=0.20
SINGLE: 합=1.00, 시그니처 요소=role_semantic=0.35


In [6]:
# NDCG@K 함수
def ndcg_at_k(rels, k=10):
    rels = np.asarray(rels, dtype=float)
    if len(rels) == 0:
        return 0.0
    rels_k = rels[:k]
    gains = (2**rels_k - 1) / np.log2(np.arange(2, len(rels_k)+2))
    dcg = gains.sum()
    ideal = np.sort(rels)[::-1][:k]
    ideal_gains = (2**ideal - 1) / np.log2(np.arange(2, len(ideal)+2))
    idcg = ideal_gains.sum()
    return dcg/idcg if idcg > 0 else 0.0

# 테스트
print('테스트 NDCG@10:')
print(f'  perfect [4,4,3,3,2,2,1,1,0,0]: {ndcg_at_k([4,4,3,3,2,2,1,1,0,0], 10):.4f}')
print(f'  reverse [0,0,1,1,2,2,3,3,4,4]: {ndcg_at_k([0,0,1,1,2,2,3,3,4,4], 10):.4f}')

테스트 NDCG@10:
  perfect [4,4,3,3,2,2,1,1,0,0]: 1.0000
  reverse [0,0,1,1,2,2,3,3,4,4]: 0.4889


In [7]:
# 25셀 매트릭스 계산 (가중치 6세트 × 평가 데이터셋 5관점)
# 메트릭: NDCG@10, NDCG@5, Recall@5, MRR@10
results = []
for w_code, weights in WEIGHTS.items():
    for eval_code in 'ABCDE':
        L = labels[eval_code].copy()
        L.columns = [c.lstrip('\ufeff') for c in L.columns]
        L['job_id'] = L['job_id'].astype(str)
        # fusion 6요소 join
        M = L.merge(
            wr[['userId', 'job_id'] + fusion_cols],
            on=['userId', 'job_id'], how='left', suffixes=('_label', '')
        )
        # 미매칭은 0으로 (보수적; 해당 후보는 사실상 fusion 점수 없음)
        for col in fusion_cols:
            if col in M.columns:
                M[col] = M[col].fillna(0)
            else:
                M[col] = 0
        # 새 가중치로 점수 계산
        M['new_score'] = sum(M[col] * weights[col] for col in fusion_cols)
        # 사용자별 그룹화 → 재정렬 → NDCG
        ndcgs10, ndcgs5, recall5, mrr10 = [], [], [], []
        for uid, g in M.groupby('userId'):
            g_sorted = g.sort_values('new_score', ascending=False)
            rels = g_sorted['judge_relevance'].fillna(0).values
            ndcgs10.append(ndcg_at_k(rels, 10))
            ndcgs5.append(ndcg_at_k(rels, 5))
            # Recall@5 = sum(rel >= 3 in top5) / sum(rel >= 3 in all)
            rels_pos = (rels >= 3).astype(int)
            total_pos = rels_pos.sum()
            recall5.append((rels_pos[:5].sum() / total_pos) if total_pos > 0 else 0.0)
            # MRR@10: 1/(rank of first rel>=3)
            first_pos = np.where(rels_pos[:10] == 1)[0]
            mrr10.append(1/(first_pos[0]+1) if len(first_pos) > 0 else 0.0)
        results.append({
            'weight_set': w_code,
            'eval_dataset': eval_code,
            'NDCG@10': float(np.mean(ndcgs10)),
            'NDCG@5': float(np.mean(ndcgs5)),
            'Recall@5': float(np.mean(recall5)),
            'MRR@10': float(np.mean(mrr10)),
            'N_users': len(ndcgs10),
            'N_pairs': len(M),
        })

results_df = pd.DataFrame(results)
print(results_df.round(4).to_string(index=False))

weight_set eval_dataset  NDCG@10  NDCG@5  Recall@5  MRR@10  N_users  N_pairs
         A            A   0.9794  0.9421    0.6016  1.0000       10      100
         A            B   0.9352  0.8762    0.5944  0.7700       10      100
         A            C   0.9227  0.8348    0.5782  0.8333       10      100
         A            D   0.9703  0.9043    0.5520  0.9000       10      100
         A            E   0.9540  0.9191    0.7472  0.8833       10      100
         B            A   0.9776  0.9402    0.6016  1.0000       10      100
         B            B   0.9323  0.8728    0.5944  0.7700       10      100
         B            C   0.9359  0.8515    0.5782  0.8833       10      100
         B            D   0.9564  0.8885    0.5520  0.8333       10      100
         B            E   0.9614  0.9284    0.7472  0.8833       10      100
         C            A   0.9685  0.9302    0.6016  0.9500       10      100
         C            B   0.9338  0.8745    0.5944  0.7750       10      100

In [8]:
# 25셀 피벗 매트릭스
ndcg10_matrix = results_df.pivot(index='weight_set', columns='eval_dataset', values='NDCG@10')
ndcg10_matrix = ndcg10_matrix.loc[['A','B','C','D','E','SINGLE'], ['A','B','C','D','E']]
ndcg10_matrix['mean'] = ndcg10_matrix.mean(axis=1)
print('=== 25셀 NDCG@10 매트릭스 ===')
print(ndcg10_matrix.round(4).to_string())
ndcg10_matrix.to_csv(OUT_DIR / '25cell_ndcg10.csv')

ndcg5_matrix = results_df.pivot(index='weight_set', columns='eval_dataset', values='NDCG@5')
ndcg5_matrix = ndcg5_matrix.loc[['A','B','C','D','E','SINGLE'], ['A','B','C','D','E']]
ndcg5_matrix['mean'] = ndcg5_matrix.mean(axis=1)
print('\n=== 25셀 NDCG@5 매트릭스 ===')
print(ndcg5_matrix.round(4).to_string())
ndcg5_matrix.to_csv(OUT_DIR / '25cell_ndcg5.csv')

=== 25셀 NDCG@10 매트릭스 ===
eval_dataset       A       B       C       D       E    mean
weight_set                                                  
A             0.9794  0.9352  0.9227  0.9703  0.9540  0.9523
B             0.9776  0.9323  0.9359  0.9564  0.9614  0.9527
C             0.9685  0.9338  0.9359  0.9564  0.9614  0.9512
D             0.9776  0.9352  0.9227  0.9703  0.9750  0.9562
E             0.9776  0.9352  0.9227  0.9703  0.9622  0.9536
SINGLE        0.9794  0.9367  0.9227  0.9564  0.9614  0.9513

=== 25셀 NDCG@5 매트릭스 ===
eval_dataset       A       B       C       D       E    mean
weight_set                                                  
A             0.9421  0.8762  0.8348  0.9043  0.9191  0.8953
B             0.9402  0.8728  0.8515  0.8885  0.9284  0.8963
C             0.9302  0.8745  0.8515  0.8885  0.9284  0.8946
D             0.9402  0.8762  0.8348  0.9043  0.9440  0.8999
E             0.9402  0.8762  0.8348  0.9043  0.9294  0.8970
SINGLE        0.9421  0.8779  0.834

In [9]:
# 가설 검증
print('=== H1: 대각선 1위 검증 (관점 i 가중치가 관점 i 라벨에서 최선?) ===')
for p in 'ABCDE':
    col = ndcg10_matrix[p].drop('mean', errors='ignore').sort_values(ascending=False)
    winner = col.index[0]
    diag_val = ndcg10_matrix.loc[p, p]
    best_val = col.iloc[0]
    is_diag = '✅' if winner == p else '❌'
    print(f'관점 {p}: 1위={winner}({best_val:.4f}) vs 대각선 {p}({diag_val:.4f}) {is_diag}')

print('\n=== H2: D 관점 industry=0.55 효과 ===')
d_single = ndcg10_matrix.loc['SINGLE', 'D']
d_d = ndcg10_matrix.loc['D', 'D']
print(f'단일 가중치(industry=0.10) NDCG@10[D]={d_single:.4f}')
print(f'D세트(industry=0.55) NDCG@10[D]={d_d:.4f}')
print(f'Δ = {d_d - d_single:+.4f}p')

print('\n=== H3: 5관점 평균 ===')
diag_mean = np.mean([ndcg10_matrix.loc[p, p] for p in 'ABCDE'])
single_mean = ndcg10_matrix.loc['SINGLE', 'mean']
print(f'5세트 분기 대각선 평균 NDCG@10 = {diag_mean:.4f}')
print(f'단일 가중치 5관점 평균 NDCG@10 = {single_mean:.4f}')
print(f'Δ = {diag_mean - single_mean:+.4f}p')

=== H1: 대각선 1위 검증 (관점 i 가중치가 관점 i 라벨에서 최선?) ===
관점 A: 1위=A(0.9794) vs 대각선 A(0.9794) ✅
관점 B: 1위=SINGLE(0.9367) vs 대각선 B(0.9323) ❌
관점 C: 1위=B(0.9359) vs 대각선 C(0.9359) ❌
관점 D: 1위=A(0.9703) vs 대각선 D(0.9703) ❌
관점 E: 1위=D(0.9750) vs 대각선 E(0.9622) ❌

=== H2: D 관점 industry=0.55 효과 ===
단일 가중치(industry=0.10) NDCG@10[D]=0.9564
D세트(industry=0.55) NDCG@10[D]=0.9703
Δ = +0.0139p

=== H3: 5관점 평균 ===
5세트 분기 대각선 평균 NDCG@10 = 0.9560
단일 가중치 5관점 평균 NDCG@10 = 0.9513
Δ = +0.0047p


In [10]:
# 결과 저장
results_df.to_csv(OUT_DIR / 'exp006_results_long.csv', index=False)

summary = {
    'exp_id': 'exp-006',
    'title': '5세트 fusion 가중치 × 5관점 라벨 25셀 매트릭스',
    'date': '2026-05-18',
    'data_source': 'raw/data/gemini_profile_outputs/{benchmark_labeled_100_X.csv, weighted_results.csv}',
    'n_perspectives': 5,
    'n_weight_sets': 6,
    'n_pairs_per_perspective': 100,
    'metrics': {
        'diagonal_winners': {p: ndcg10_matrix[p].drop('mean', errors='ignore').idxmax() for p in 'ABCDE'},
        'diagonal_ndcg10': {p: float(ndcg10_matrix.loc[p, p]) for p in 'ABCDE'},
        'single_ndcg10': {p: float(ndcg10_matrix.loc['SINGLE', p]) for p in 'ABCDE'},
        'avg_diagonal_ndcg10': float(diag_mean),
        'avg_single_ndcg10': float(single_mean),
        'delta_diagonal_vs_single': float(diag_mean - single_mean),
        'D_perspective_industry_effect': float(d_d - d_single),
    }
}
with open(OUT_DIR / 'exp006_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print(f'\nsaved: {OUT_DIR}')
print(json.dumps(summary['metrics'], ensure_ascii=False, indent=2))


saved: raw/experiments/exp-006-25cell-matrix
{
  "diagonal_winners": {
    "A": "A",
    "B": "SINGLE",
    "C": "B",
    "D": "A",
    "E": "D"
  },
  "diagonal_ndcg10": {
    "A": 0.9793575633440395,
    "B": 0.9322938896820434,
    "C": 0.935921410702645,
    "D": 0.970343665501672,
    "E": 0.9621921485722321
  },
  "single_ndcg10": {
    "A": 0.9793575633440395,
    "B": 0.9367249910106381,
    "C": 0.9226709526337592,
    "D": 0.9564046605799732,
    "E": 0.9613632205124331
  },
  "avg_diagonal_ndcg10": 0.9560217355605264,
  "avg_single_ndcg10": 0.9513042776161684,
  "delta_diagonal_vs_single": 0.004717457944357961,
  "D_perspective_industry_effect": 0.01393900492169875
}
